In [4]:
import sys

sys.path.append("..")

import shap
import numpy as np
import pandas as pd
from data import create_data_for_lstm
from seq2seq import encode_decode_multitask

ModuleNotFoundError: No module named 'processing'

In [ ]:
decoder_path = f"/home/aevans/nwp_bias/src/machine_learning/data/parent_models/HRRR/s2s/Hudson Valley/Hudson Valley_t2m_VOOR_encoder.pth"
encoder_path = f"/home/aevans/nwp_bias/src/machine_learning/data/parent_models/HRRR/s2s/Hudson Valley/Hudson Valley_t2m_VOOR_decoder.pth"

In [ ]:
model = encode_decode_multitask.ShallowLSTM_seq2seq_multi_task(
    num_sensors=num_sensors,
    hidden_units=hidden_units,
    num_layers=num_layers,
    mlp_units=1500,
    device=device,
    num_stations=len(stations),
)
if os.path.exists(encoder_path):
    print("Loading Encoder Model")
    model.encoder.load_state_dict(torch.load(encoder_path), strict=False)
    # Example usage for encoder and decoder
    get_model_file_size(encoder_path)

if os.path.exists(decoder_path):
    print("Loading Decoder Model")
    model.decoder.load_state_dict(torch.load(decoder_path), strict=False)
    get_model_file_size(decoder_path)

In [ ]:
(df_train, df_test, df_val, features, stations, target, vt, _) = (
    create_data_for_lstm.create_data_for_model(station, fh, today_date, metvar)
)

In [ ]:
# --- Step 2: Create Kernel SHAP Explainer --
background = X_test[:100].reshape(100, -1)  # [n_samples, seq_len * n_features]
explainer = shap.KernelExplainer(model.predict(), background)

In [ ]:
X_sample = X_test[:10].reshape(10, -1)  # flatten
shap_values = explainer.shap_values(X_sample)
shap.summary_plot(shap_values, X_sample)

In [ ]:
# --- Step 5: Visualize for one sample ---
sample_id = 0
plt.figure(figsize=(10, 6))
sns.heatmap(
    shap_array[sample_id],
    cmap="coolwarm",
    xticklabels=feature_names,
    yticklabels=range(seq_len),
)
plt.title(f"SHAP Values (Sample {sample_id})")
plt.xlabel("Features")
plt.ylabel("Time Step")
plt.tight_layout()
plt.show()

# Optional: SHAP summary plot across all samples (flattened again)
shap.summary_plot(
    shap_values,
    X_sample,
    feature_names=[f"{f}_t{t}" for t in range(seq_len) for f in feature_names],
)